## 문제 해결에 앞서서 — Dataset 만들기

#### 1. apple이 포함된 문장 200개 생성
1) 첫번째 시도 — AI를 통해 데이터셋을 생성

생성형 AI(GPT, Gemini)를 활용하여 데이터셋을 생성했지만, 정말 ‘자연적인’ 데이터셋이라기 보단, 의도된 데이터셋에 가까웠다. 

예시:

Sentence,Meaning,Tag
I had an apple with breakfast.,과일,CNN
The green apple tasted sour.,과일,CNN
She put the apple in the refrigerator.,과일,CNN
An apple a day is a common saying.,과일,CNN
We used apple slices in the salad.,과일,CNN
Apple unveiled a new MacBook.,회사,PNN

사람이 사용하는, 문법이 조금 무너지거나 노이즈 섞인 긴 문장이라기 보단 교과서적인, 사전적인 예문에 가까운 문장을 생성했다. 따라서 실제로 인간이 작성한 문장을 수집하기로 했다. 

2) 두번째 시도 — 실제 문장들을 수집

인간이 작성한 문장을 수집하기 위해서 

In [5]:
"""Build a reproducible, source-traceable apple dataset from Wikimedia pages.

This script DOES NOT generate sentences.  It retrieves existing English-language
Wikimedia article text through the MediaWiki API, extracts sentences containing
the exact word apple/Apple, and assigns a provisional label from the source topic.

Run in Google Colab or another environment with internet access:
    !python collect_authentic_wikimedia_dataset.py

Outputs
-------
authentic_candidates.csv : 200 examples with source, licence and provenance
train_authentic.csv      : 160 examples (CNN 80 / PNN 80)
test_authentic.csv       :  40 examples (CNN 20 / PNN 20)
audit_sample.csv         :  20 examples to check manually before submission

The source-title label is only a first-pass annotation.  Read the 20 audit rows,
and correct/exclude any sentence whose referent is not the expected fruit/company.
"""

from __future__ import annotations

import csv
import json
import random
import re
import time
from collections import defaultdict
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen


# ``__file__`` exists when this runs as a .py file, but not when users paste the
# code into a Colab/Jupyter cell.  In a notebook, save output in the current
# working directory instead.
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "authentic_wikimedia_dataset"
RANDOM_SEED = 20260920
TARGET_PER_LABEL = 100
MAX_PER_SOURCE = 12
LICENSE = "CC BY-SA 4.0"
API_URL = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "Mozilla/5.0 (compatible; apple-pos-course-project/1.0; educational use)"

# Each group is deliberately broad: no sentence is written or paraphrased here.
# The topic itself provides a provisional sense label that must later be audited.
SOURCE_TITLES = {
    "CNN": [
        "Apple", "Apple pie", "Apple cider", "Apple juice", "Apple sauce",
        "Apple butter", "Apple crisp", "Apple dumpling", "Apple strudel",
        "Apple cake", "Apple production", "Apple cultivar", "Apple tree",
        "Apple orchard", "Baked apple", "Apple bobbing", "Apple turnover",
        "Apple fritter", "Apple crumble", "Apple cobbler", "Apple galette",
        "Apple cheese", "Apple jelly", "Caramel apple", "Candied apple",
        "Apple brandy", "Apple chips", "Apple bread", "Apple fritter",
    ],
    "PNN": [
        "Apple Inc.", "IPhone", "IPad", "Macintosh", "MacBook", "Apple Watch",
        "Apple TV", "Apple Music", "Apple Pay", "Apple Store", "Apple silicon",
        "IOS", "AirPods", "Apple Vision Pro", "App Store (Apple)",
        "Apple Card", "Apple Maps", "Apple Arcade", "Apple News", "Apple Books",
        "ICloud", "Apple Podcasts", "Apple Intelligence", "Apple Wallet", "Apple One",
        "Apple TV+", "Apple Studio Display", "Apple Pro Display XDR",
        "Apple A series", "Apple Developer", "List of Apple products",
    ],
}


def api_json(params: dict[str, str]) -> dict:
    """Fetch one JSON response from MediaWiki and preserve actionable errors."""
    full_params = {"format": "json", "formatversion": "2", "origin": "*", **params}
    request = Request(
        f"{API_URL}?{urlencode(full_params)}",
        headers={"User-Agent": USER_AGENT, "Accept": "application/json"},
    )
    try:
        with urlopen(request, timeout=30) as response:  # nosec B310 - fixed HTTPS endpoint
            data = json.load(response)
    except HTTPError as error:
        raise RuntimeError(f"HTTP {error.code}: {error.reason}") from error
    except URLError as error:
        raise RuntimeError(f"Network error: {error.reason}") from error
    if "error" in data:
        raise RuntimeError(f"MediaWiki API error: {data['error'].get('info', data['error'])}")
    return data


def get_page_extract(title: str) -> tuple[str, str] | None:
    """Return canonical title and plaintext extract, or None if the page is absent."""
    data = api_json({
        "action": "query", "format": "json", "redirects": "1",
        "prop": "extracts", "explaintext": "1", "titles": title,
    })
    pages = data.get("query", {}).get("pages", [])
    page = pages[0] if isinstance(pages, list) and pages else next(iter(pages.values()), {})
    if "missing" in page:
        return None
    extract = page.get("extract", "").strip()
    canonical_title = page.get("title", title)
    return (canonical_title, extract) if extract else None


def split_sentences(text: str) -> list[str]:
    """A conservative splitter; it avoids pulling headings/bullets into examples."""
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return re.split(r"(?<=[.!?])\s+(?=[A-Z\"'])", text)


def clean_candidate(sentence: str) -> str | None:
    sentence = sentence.strip()
    if not re.search(r"\bapple(?:'s)?\b", sentence, flags=re.IGNORECASE):
        return None
    if not 45 <= len(sentence) <= 350:
        return None
    if sentence.count(";") > 2 or sentence.count(":") > 2:
        return None
    return sentence


def collect_candidates(label: str) -> list[dict[str, str]]:
    """Collect de-duplicated candidate sentences, preserving their exact source."""
    candidates: list[dict[str, str]] = []
    seen: set[str] = set()
    failures: list[str] = []
    successful_sources = 0
    for requested_title in SOURCE_TITLES[label]:
        try:
            response = get_page_extract(requested_title)
        except Exception as error:  # network failure should not discard completed sources
            failures.append(f"{requested_title}: {error}")
            continue
        if response is None:
            print(f"Skipping missing page: {requested_title}")
            continue
        title, extract = response
        successful_sources += 1
        source_url = "https://en.wikipedia.org/wiki/" + quote(title.replace(" ", "_"))
        used_from_source = 0
        for sentence in split_sentences(extract):
            sentence = clean_candidate(sentence)
            normalised = re.sub(r"\W+", "", sentence.lower()) if sentence else ""
            if not sentence or normalised in seen:
                continue
            seen.add(normalised)
            candidates.append({
                "sentence": sentence,
                "label": label,
                "source_title": title,
                "source_url": source_url,
                "source_license": LICENSE,
                "label_note": f"Provisional {label}: sentence retrieved from the '{title}' topic page.",
            })
            used_from_source += 1
            if used_from_source >= MAX_PER_SOURCE:
                break
        time.sleep(0.2)  # polite request rate
    print(f"{label}: {successful_sources} source pages retrieved, {len(candidates)} candidates found")
    if not candidates and failures:
        raise RuntimeError("No candidate was retrieved. First API failure: " + failures[0])
    return candidates


def choose_diverse_examples(candidates: list[dict[str, str]], target: int, rng: random.Random) -> list[dict[str, str]]:
    """Sample in source-title rounds so one long article cannot dominate the data."""
    by_source: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in candidates:
        by_source[row["source_title"]].append(row)
    for rows in by_source.values():
        rng.shuffle(rows)

    chosen: list[dict[str, str]] = []
    while len(chosen) < target:
        progressed = False
        for title in sorted(by_source):
            if by_source[title] and len(chosen) < target:
                chosen.append(by_source[title].pop())
                progressed = True
        if not progressed:
            break
    if len(chosen) < target:
        raise RuntimeError(f"Only found {len(chosen)} usable examples; need {target}.")
    return chosen


def group_split(rows: list[dict[str, str]], test_count: int, rng: random.Random) -> tuple[list[dict[str, str]], list[dict[str, str]]]:
    """Hold out whole source pages where possible, then fill to the exact test size."""
    by_source: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        by_source[row["source_title"]].append(row)
    titles = list(by_source)
    rng.shuffle(titles)
    test: list[dict[str, str]] = []
    train: list[dict[str, str]] = []
    for title in titles:
        group = by_source[title]
        if len(test) + len(group) <= test_count:
            test.extend(group)
        else:
            train.extend(group)
    # Exact balance is more important than a perfect group boundary for a small PBL dataset.
    rng.shuffle(train)
    while len(test) < test_count:
        test.append(train.pop())
    while len(test) > test_count:
        train.append(test.pop())
    return train, test


def write_csv(path: Path, rows: list[dict[str, str]], columns: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as file:
        # Detailed provenance stays in authentic_candidates.csv, while the
        # train/test files intentionally expose only sentence and label.
        writer = csv.DictWriter(file, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def main() -> None:
    rng = random.Random(RANDOM_SEED)
    OUTPUT_DIR.mkdir(exist_ok=True)
    selected: dict[str, list[dict[str, str]]] = {}
    for label in ("CNN", "PNN"):
        pool = collect_candidates(label)
        print(f"{label}: found {len(pool)} usable candidates")
        selected[label] = choose_diverse_examples(pool, TARGET_PER_LABEL, rng)

    train_rows: list[dict[str, str]] = []
    test_rows: list[dict[str, str]] = []
    for label in ("CNN", "PNN"):
        train, test = group_split(selected[label], test_count=20, rng=rng)
        train_rows.extend(train)
        test_rows.extend(test)
    rng.shuffle(train_rows)
    rng.shuffle(test_rows)

    detailed_columns = ["sentence", "label", "source_title", "source_url", "source_license", "label_note"]
    write_csv(OUTPUT_DIR / "authentic_candidates.csv", train_rows + test_rows, detailed_columns)
    write_csv(OUTPUT_DIR / "train_authentic.csv", train_rows, ["sentence", "label"])
    write_csv(OUTPUT_DIR / "test_authentic.csv", test_rows, ["sentence", "label"])

    audit = rng.sample(train_rows + test_rows, k=20)
    write_csv(OUTPUT_DIR / "audit_sample.csv", audit, detailed_columns)
    report = [
        "Authentic apple dataset collection report",
        f"Source: English Wikipedia MediaWiki API ({LICENSE})",
        "No sentences were generated or paraphrased by AI.",
        "Labels are provisional source-topic labels and require the 20-row audit before submission.",
        f"Train: {len(train_rows)} rows; Test: {len(test_rows)} rows.",
        f"Train balance: CNN={sum(r['label'] == 'CNN' for r in train_rows)}, PNN={sum(r['label'] == 'PNN' for r in train_rows)}.",
        f"Test balance: CNN={sum(r['label'] == 'CNN' for r in test_rows)}, PNN={sum(r['label'] == 'PNN' for r in test_rows)}.",
    ]
    (OUTPUT_DIR / "collection_report.txt").write_text("\n".join(report) + "\n", encoding="utf-8")
    print("Done. Review audit_sample.csv before using the train/test files.")


if __name__ == "__main__":
    main()

CNN: 11 source pages retrieved, 120 candidates found
CNN: found 120 usable candidates
PNN: 11 source pages retrieved, 117 candidates found
PNN: found 117 usable candidates
Done. Review audit_sample.csv before using the train/test files.


## 문제 1: 규칙 기반 품사 태거 만들기

In [6]:
import os
import re
import pandas as pd

def rule_based_apple_tagger(sentence):
    """
    위키미디어 실제 문장에서 'apple'의 품사를 판별하는 규칙 기반 태거
    - CNN: 일반명사 (과일/식품)
    - PNN: 고유명사 (기업/IT)
    """
    tokens = re.findall(r"\b[\w']+\b", sentence)
    
    # 규칙 1: 대소문자 및 위치 규칙 (문장 중간/끝의 대문자 Apple/Apple's -> PNN)
    for i, token in enumerate(tokens):
        if token in ["Apple", "Apple's"]:
            if i > 0:  # 문장 첫 단어가 아닌 대문자 Apple은 기업(PNN)
                return "PNN"
        elif token in ["apple", "apples", "apple's"]:
            return "CNN"
            
    # 규칙 2: 구문 및 관사 결합 규칙 (a/an + apple -> CNN)
    for i, token in enumerate(tokens):
        if token.lower() in ["apple", "apples"]:
            if i > 0 and tokens[i-1].lower() in ["an", "a"]:
                return "CNN"
            if i > 1 and tokens[i-2].lower() in ["an", "a"]:
                return "CNN"
                
    # 규칙 3: 문맥 어휘 사전(Semantic Context Lexicon) 도메인 스코어링
    company_keywords = {
        "inc", "corp", "corporation", "software", "app", "apps", "iphone", "ipad", 
        "macbook", "mac", "stock", "shares", "store", "ceo", "founder", "headquarters", 
        "cupertino", "system", "updated", "released", "announced", "new", "laptop", 
        "tech", "technology", "device", "event", "revenue", "quarter", "silicon"
    }
    fruit_keywords = {
        "ate", "eat", "eaten", "bought", "buy", "pie", "cider", "juice", "sweet", 
        "sour", "crisp", "red", "green", "tree", "orchard", "peeled", "slice", 
        "slices", "salad", "fresh", "delicious", "wash", "washed", "basket", "bite", 
        "autumn", "flavor", "cultivar", "dessert", "fruit", "harvest"
    }
    
    words = set(re.findall(r"\b\w+\b", sentence.lower()))
    comp_score = len(words.intersection(company_keywords))
    fruit_score = len(words.intersection(fruit_keywords))
    
    if comp_score > fruit_score:
        return "PNN"
    elif fruit_score > comp_score:
        return "CNN"
        
    # 예외 처리: 문장 첫 단어가 대문자 "Apple"이고 키워드가 없는 경우 PNN 기본값
    if tokens and tokens[0] == "Apple":
        return "PNN"
        
    return "CNN"

def main():
    # 위키미디어 생성 폴더 경로 자동 설정
    dataset_dir = "authentic_wikimedia_dataset"
    test_path = os.path.join(dataset_dir, "test_authentic.csv")
    
    if not os.path.exists(test_path):
        test_path = "test_authentic.csv"  # 현재 디렉토리에 파일이 있는 경우 예외 처리
        
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"테스트 데이터셋을 찾을 수 없습니다: {test_path}\n"
                                f"먼저 collect_authentic_wikimedia_dataset.py를 실행해주세요.")
        
    test_df = pd.read_csv(test_path)
    
    # 위키미디어 데이터셋 컬럼명('sentence', 'label') 적용
    test_df['predicted_label'] = test_df['sentence'].apply(rule_based_apple_tagger)
    
    accuracy = (test_df['label'] == test_df['predicted_label']).mean()
    print("=== [문제 1] 위키미디어 데이터 기반 규칙 태거 평가 ===")
    print(f"테스트 데이터셋 경로: {test_path}")
    print(f"총 평가 문장 수: {len(test_df)}")
    print(f"테스트 정확도: {accuracy * 100:.2f}%\n")

if __name__ == "__main__":
    main()

=== [문제 1] 위키미디어 데이터 기반 규칙 태거 평가 ===
테스트 데이터셋 경로: authentic_wikimedia_dataset/test_authentic.csv
총 평가 문장 수: 40
테스트 정확도: 97.50%



## 문제 2: 기계학습 기반 품사 태거 만들기

In [7]:
import os
import re
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

def extract_features(sentence):
    """
    위키미디어 문장에서 'apple' 관련 위치, 대소문자, 주변 문맥 어휘 특성(Feature) 추출
    """
    tokens = re.findall(r"\b[\w']+\b", sentence)
    
    apple_idx = -1
    apple_token = ""
    for i, t in enumerate(tokens):
        if t.lower() in ['apple', "apple's", 'apples']:
            apple_idx = i
            apple_token = t
            break
            
    # 특성 1: Target 단어의 대소문자 여부 (Capitalization)
    is_capitalized = apple_token.istitle() if apple_token else False
    
    # 특성 2: Target 단어가 문장 첫 단어인지 여부
    is_first_word = (apple_idx == 0)
    
    # 특성 3: 바로 앞/뒤 단어 (Grammatical Context)
    prev_word = tokens[apple_idx - 1].lower() if apple_idx > 0 else "<START>"
    next_word = tokens[apple_idx + 1].lower() if apple_idx < len(tokens) - 1 else "<END>"
    
    # 특성 4: Target 제외 주변 문맥 단어 (Bag of Context Words)
    context_words = [t.lower() for i, t in enumerate(tokens) if i != apple_idx]
    
    features = {
        'is_capitalized': is_capitalized,
        'is_first_word': is_first_word,
        'prev_word': prev_word,
        'next_word': next_word,
    }
    for w in context_words:
        features[f'context_{w}'] = True
        
    return features

def main():
    dataset_dir = "authentic_wikimedia_dataset"
    train_path = os.path.join(dataset_dir, "train_authentic.csv")
    test_path = os.path.join(dataset_dir, "test_authentic.csv")
    
    if not os.path.exists(train_path):
        train_path = "train_authentic.csv"
        test_path = "test_authentic.csv"
        
    if not os.path.exists(train_path):
        raise FileNotFoundError("학습 및 테스트 데이터셋을 찾을 수 없습니다.\n"
                                "먼저 collect_authentic_wikimedia_dataset.py를 실행해주세요.")
        
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # 위키미디어 데이터셋 컬럼명('sentence', 'label') 적용
    X_train_dict = [extract_features(s) for s in train_df['sentence']]
    X_test_dict = [extract_features(s) for s in test_df['sentence']]
    
    # DictVectorizer를 이용한 원-핫 인코딩 수치화
    vec = DictVectorizer(sparse=False)
    X_train = vec.fit_transform(X_train_dict)
    X_test = vec.transform(X_test_dict)
    
    y_train = train_df['label']
    y_test = test_df['label']
    
    # 선형 SVM 모델 학습
    model = SVC(kernel='linear', C=1.0, random_state=42)
    model.fit(X_train, y_train)
    
    # 예측 및 평가
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print("=== [문제 2] 위키미디어 데이터 기반 기계학습(SVM) 태거 평가 ===")
    print(f"학습 데이터 수: {len(train_df)}, 테스트 데이터 수: {len(test_df)}")
    print(f"테스트 정확도: {accuracy * 100:.2f}%\n")
    print("[상세 분류 리포트]")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    main()

=== [문제 2] 위키미디어 데이터 기반 기계학습(SVM) 태거 평가 ===
학습 데이터 수: 160, 테스트 데이터 수: 40
테스트 정확도: 92.50%

[상세 분류 리포트]
              precision    recall  f1-score   support

         CNN       1.00      0.85      0.92        20
         PNN       0.87      1.00      0.93        20

    accuracy                           0.93        40
   macro avg       0.93      0.93      0.92        40
weighted avg       0.93      0.93      0.92        40



## 문제 3: 모델 평가 및 성능 비교
#### 1. 테스트 데이터셋 평과 결과

위피키디아 실제 문서에서 추출한 테스트 데이터셋 대상으로 수행한 평가 결과는 이렇다:

- 규칙 기반 태거
    - 분류 모델/ 규칙 구성: 대소문자/위치 규칙 + 구문 결합 + 문맥 어휘 사전 스코어링
    - 테스트 정확도: 97.50%
    - 맞힌 개수 / 전체: 39 / 40
- 기계학습 기반 태거
    - 분류 모델/ 규칙 구성: Target 중심 특징 공학 (Feature Engineering) + 선형 SVM
    - 테스트 정확도 92.50%
    - 맞힌 개수 / 전체: 37 / 40

#### 2. 오답 분석 및 성능 특성 비교
- 규칙 기반 태거 (정확도 97.50%)
    - 40개 테스트 문장 중 단 1개만 오분류 (39개 정답).
    - 위키피디아 도메인 어휘(cider, orchard, stock, revenue 등) 사전과 대소문자 위치 예외 규칙이 잘 통합되어 매우 높은 정확도를 나타냈다. 

- 기계학습(SVM) 기반 태거 (정확도 92.50%)
    - CNN (일반명사): Precision 1.00, Recall 0.85 (20개 중 17개 정답, 3개 오답)
    - PNN (고유명사): Precision 0.87, Recall 1.00 (20개 중 20개 정답, 0개 오답)
    - 오류 패턴 원인 분석:
    PNN 편향 (Over-prediction towards PNN): PNN의 Recall은 1.00인 반면, CNN의 Recall은 0.85에 머물렀다. 이는 모델이 모호한 문장을 PNN(기업/IT)으로 예측하는 경향이 강함을 의미한다.
    - 대소문자 및 소유격 특성의 강력한 가중치: 위키피디아 문서 특성상 대문자 Apple이나 소유격 Apple's 표기가 학습 데이터(160개)에서 PNN 클래스와 매우 높은 상관관계를 형성했다. 그 결과 문장 첫 단어로 출현한 대문자 과일 표현("Apple pie is...")이나 특수 구문이 포함된 CNN 문장 3개가 PNN으로 잘못 오분류되었다.

## 문제 4: 규칙 기반 vs 기계학습 기반 비교 분석
#### 1. 개발 프로세스의 차이
- 규칙 기반: 사람이 직접 위키피디아 문장의 언어적 패턴(관사 an, 대소문자 위치, 도메인 키워드)을 관찰하고 명시적인 조건문(if-else)을 수동으로 구축한다.

- 기계학습 기반: 사람이 문장에서 위치, 대소문자, 주변 단어 등 특징(Feature)의 구조만 정의하고, 선형 SVM 알고리즘이 160개 학습 데이터로부터 각 특징의 최적 가중치(Weight)를 자동으로 학습한다.

#### 2. 장단점 및 특성 비교
- 설명 가능성: 
    - 규칙 기반 태거: 매우 높음; 특정 규칙 조건문으로 오분류 원인을 즉시 파악 및 수정 가능
    - 기계학습 기반 태거: 낮음; 특징 가중치 간 복잡한 상호작용으로 개별 예측 원인 추적이 비교적 어려움
- 데이터 의존성:
    - 규칙 기반 태거: 낮음; 학습 데이터 없이 정교한 규칙 설계만으로 높은 초기 성능 확보 가능
    - 기계학습 기반 태거: 높음; 양질의 레이블링 데이터(train_authentic.csv)가 필수적임
- 일반화 능력: 
    - 규칙 기반 태거: 제한적; 규칙에 정의되지 않은 신규 어휘나 비표준 문장 구조에서 성능 저하
    - 기계학습 기반 태거: 우수; 미처 예견하지 못한 문맥도 수치적 유사성에 기반하여 유연하게 판별
- 유지보수 및 확장성:
    - 규칙 기반 태거: 어려움; 예외 규칙이 늘어날수록 규칙 간 충돌 및 복잡도가 급증함
    - 기계학습 기반 태거: 용이함; 신규 데이터 추가 후 재학습(Retraining)만으로 모델 업데이트 가능

## 문제 5: 성능 향상을 위한 개선 방안

#### 1. 학습 데이터셋 전처리 및 특수 토큰 처리 (Text Normalization)

소유격(Apple's), 기호, 대소문자 변환 로직을 정교화한다. 문장 첫 단어의 경우 무조건적인 대문자 특성 부여를 지양하고, 표제어 추출(Lemmatization)을 함께 수행한다.
#### 2. (형태소/품사(POS) 및 N-gram 피처 고도화

NLTK의 pos_tag를 도입하여 Target 단어 앞뒤의 품사 정보(예: DT + NN 구조)를 특징 벡터에 추가하고, 단일 단어(Unigram) 외에 2개 단어 조합(Bigram) 문맥 피처를 확장한다.

#### 3. 하이브리드(Hybrid) 앙상블 시스템 구축

1차로 명확한 관사/구문 규칙(an apple $\rightarrow$ CNN, Apple Inc. $\rightarrow$ PNN)을 적용하고, 규칙으로 판별하기 모호한 문장에 대해서만 SVM의 예측 확률값(Decision function)을 참조하는 하이브리드 분류 구조를 적용한다. 

## 문제 6: 대화형 AI(LLM) 상호작용 경험 및 평가

#### 1. 대화형 AI 활용의 장점
- 자동화 및 개발 속도 향상: MediaWiki API를 활용한 위키피디아 실제 데이터 추출부터 스플릿, DictVectorizer 수치화, SVM 파이프라인 구성까지 전체 과제 파이프라인을 신속하게 구축했다.

- 코드 안정성 확보: 스크립트 실행 과정에서 발생할 수 있는 파일 경로 문제, 컬럼명(sentence, label) 불일치 오류를 빠르게 검증하고 수정할 수 있었다.

#### 2. 상호작용 과정의 한계 및 문제점
- 데이터 스키마 및 환경 차이: LLM이 초기 제안한 코드와 실제 수집 스크립트(collect_authentic_wikimedia_dataset.py)가 생성하는 파일 경로/컬럼 구조 간 차이가 존재하여 추가적인 코드 조율이 필요했다.

- 데이터 특성 세부 반영의 필요성: 소유격(Apple's) 처리나 문장 첫 단어 대문자 문제와 같이 실제 위키피디아 텍스트에서 빈번히 발생하는 언어적 예외 케이스를 정교하게 다루기 위해 추가적인 특성 공학 가이드가 요구되었다.

#### 3. 향후 효율적 활용을 위한 개선 방안
- 명확한 입출력 프롬프트 제공: 데이터셋의 정확한 파일 경로, 컬럼명, 데이터 타입 및 요구되는 평가 지표를 프롬프트에 명시하여 초기 구현 오차를 줄인다.
- 단계별 모듈화 검증: [데이터 수집/전처리 $\rightarrow$ 특징 추출 $\rightarrow$ 모델 학습 $\rightarrow$ 오답 분석]의 단계별로 코드를 분할 요청하고 실행 결과를 피드백함으로써 시스템 완성도를 높인다.